# Analytics Pipeline: Data Consumption Layer for Frontend Integration
## Dashboards • Trend View • Reports • Recommendations

This notebook provides a structured analytics data pipeline designed for seamless frontend integration. It organizes analytics outputs into four main consumption layers:

1. **Dashboards**: Real-time KPIs, metrics, and summary statistics
2. **Trend View**: Time series trends, seasonality patterns, and historical analysis
3. **Reports**: Comprehensive analytics findings and model performance metrics
4. **Recommendations**: Actionable insights and prescriptive analytics outputs

### Analytics Models Covered:
- **Descriptive**: Time Series Decomposition, Clustering, Anomaly Detection
- **Predictive**: Forecasting (Prophet, SARIMA), Classification (XGBoost, Logistic Regression), State Analysis (Markov Chains)
- **Prescriptive**: Optimization (LP, MILP), Simulation (Monte Carlo), Sensitivity Analysis, Efficiency Evaluation

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Time Series and Forecasting
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing, Holt, SimpleExpSmoothing
from prophet import Prophet

# Machine Learning
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split

# XGBoost and SHAP
import xgboost as xgb
import shap

# Optimization
try:
    from pulp import *
except:
    print("PuLP not installed. Installing...")

# Visualization
from matplotlib.gridspec import GridSpec
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All libraries imported successfully!")

## 2. Generate Sample Data

In [ ]:
# Generate time series data with trend and seasonality
np.random.seed(42)
dates = pd.date_range(start='2022-01-01', end='2024-12-31', freq='D')

# Create synthetic time series with trend, seasonality, and noise
trend = np.linspace(0, 50, len(dates))
seasonality = 20 * np.sin(np.arange(len(dates)) * 2 * np.pi / 365)
noise = np.random.normal(0, 5, len(dates))
time_series = 100 + trend + seasonality + noise

# Create dataframe
df_ts = pd.DataFrame({
    'ds': dates,
    'y': time_series,
    'Volume': np.abs(np.random.normal(1000, 200, len(dates))),
    'Price': 50 + trend/2 + noise/2
})

# Generate business data for clustering and classification
np.random.seed(42)
n_customers = 500

customer_data = pd.DataFrame({
    'CustomerID': np.arange(n_customers),
    'Spending': np.random.exponential(100, n_customers),
    'Visits': np.random.poisson(10, n_customers),
    'Days_Since_Purchase': np.random.exponential(30, n_customers),
    'Churn': np.random.binomial(1, 0.2, n_customers),
    'Age': np.random.normal(40, 15, n_customers),
    'Account_Age': np.random.exponential(50, n_customers)
})

print("Time Series Data Shape:", df_ts.shape)
print("Customer Data Shape:", customer_data.shape)
print("\nFirst few rows of Time Series:")
print(df_ts.head())
print("\nFirst few rows of Customer Data:")
print(customer_data.head())

---

# DESCRIPTIVE ANALYTICS

## What is Descriptive Analytics?
Descriptive analytics examines historical data to understand patterns, trends, and distributions. It answers the question: **"What happened?"**

---

## 3. Descriptive Analytics: Time Series Decomposition

In [ ]:
# Classical Decomposition
df_ts_set = df_ts.set_index('ds')
decomposition = seasonal_decompose(df_ts_set['y'], model='additive', period=365)

# STL Decomposition for more robust results
stl = STL(df_ts_set['y'], seasonal=365, trend=91)
result_stl = stl.fit()

fig, axes = plt.subplots(4, 2, figsize=(16, 12))

# Classical Decomposition
axes[0, 0].plot(decomposition.observed, 'b-', linewidth=2)
axes[0, 0].set_ylabel('Observed', fontsize=10, fontweight='bold')
axes[0, 0].set_title('Classical Decomposition', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

axes[1, 0].plot(decomposition.trend, 'g-', linewidth=2)
axes[1, 0].set_ylabel('Trend', fontsize=10, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

axes[2, 0].plot(decomposition.seasonal, 'r-', linewidth=2)
axes[2, 0].set_ylabel('Seasonal', fontsize=10, fontweight='bold')
axes[2, 0].grid(True, alpha=0.3)

axes[3, 0].plot(decomposition.resid, 'purple', linewidth=1, alpha=0.7)
axes[3, 0].set_ylabel('Residual', fontsize=10, fontweight='bold')
axes[3, 0].set_xlabel('Date', fontsize=10, fontweight='bold')
axes[3, 0].grid(True, alpha=0.3)

# STL Decomposition
axes[0, 1].plot(result_stl.observed, 'b-', linewidth=2)
axes[0, 1].set_ylabel('Observed', fontsize=10, fontweight='bold')
axes[0, 1].set_title('STL Decomposition (Robust)', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 1].plot(result_stl.trend, 'g-', linewidth=2)
axes[1, 1].set_ylabel('Trend', fontsize=10, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

axes[2, 1].plot(result_stl.seasonal, 'r-', linewidth=2)
axes[2, 1].set_ylabel('Seasonal', fontsize=10, fontweight='bold')
axes[2, 1].grid(True, alpha=0.3)

axes[3, 1].plot(result_stl.resid, 'purple', linewidth=1, alpha=0.7)
axes[3, 1].set_ylabel('Residual', fontsize=10, fontweight='bold')
axes[3, 1].set_xlabel('Date', fontsize=10, fontweight='bold')
axes[3, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('Time Series Decomposition: Identifying Trend, Seasonality & Residuals', 
             fontsize=14, fontweight='bold', y=1.00)
plt.show()

print("✓ Time Series Decomposition Complete")
print(f"  - Trend component: Shows long-term progression")
print(f"  - Seasonal component: Shows recurring patterns")
print(f"  - Residual: Unexplained variation")

## 4. Descriptive Analytics: K-Means Clustering

In [ ]:
# Prepare data for clustering
features = ['Spending', 'Visits', 'Days_Since_Purchase']
X_cluster = customer_data[features].copy()
X_cluster_scaled = StandardScaler().fit_transform(X_cluster)

# Elbow method to find optimal clusters
inertias = []
silhouette_scores = []
from sklearn.metrics import silhouette_score

K_range = range(2, 11)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_cluster_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_cluster_scaled, kmeans.labels_))

# Optimal clusters (elbow point)
optimal_k = 3
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
customer_data['Cluster'] = kmeans_final.fit_predict(X_cluster_scaled)

fig = plt.figure(figsize=(16, 6))
gs = GridSpec(1, 3, figure=fig)

# Elbow curve
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
ax1.axvline(x=optimal_k, color='red', linestyle='--', linewidth=2, label=f'Optimal K={optimal_k}')
ax1.set_xlabel('Number of Clusters (k)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Inertia', fontsize=11, fontweight='bold')
ax1.set_title('Elbow Method', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Silhouette scores
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
ax2.axvline(x=optimal_k, color='red', linestyle='--', linewidth=2, label=f'Optimal K={optimal_k}')
ax2.set_xlabel('Number of Clusters (k)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Silhouette Score', fontsize=11, fontweight='bold')
ax2.set_title('Silhouette Analysis', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend()

# Cluster visualization (3D)
from mpl_toolkits.mplot3d import Axes3D
ax3 = fig.add_subplot(gs[0, 2], projection='3d')
scatter = ax3.scatter(X_cluster['Spending'], X_cluster['Visits'], X_cluster['Days_Since_Purchase'], 
                     c=customer_data['Cluster'], cmap='viridis', s=50, alpha=0.6)
ax3.scatter(kmeans_final.cluster_centers_[:, 0], kmeans_final.cluster_centers_[:, 1], 
           kmeans_final.cluster_centers_[:, 2], c='red', marker='X', s=300, edgecolors='black', linewidth=2)
ax3.set_xlabel('Spending', fontsize=10, fontweight='bold')
ax3.set_ylabel('Visits', fontsize=10, fontweight='bold')
ax3.set_zlabel('Days Since Purchase', fontsize=10, fontweight='bold')
ax3.set_title('K-Means Clusters (3D)', fontsize=12, fontweight='bold')
plt.colorbar(scatter, ax=ax3, label='Cluster')

plt.suptitle('K-Means Clustering: Customer Segmentation', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print("✓ K-Means Clustering Complete")
print(f"  - Identified {optimal_k} distinct customer segments")
print(f"  - Cluster distribution:\n{customer_data['Cluster'].value_counts().sort_index()}")

## 5. Descriptive Analytics: Isolation Forests (Anomaly Detection)

In [ ]:
# Apply Isolation Forest for anomaly detection
iso_forest = IsolationForest(contamination=0.1, random_state=42)
df_ts['anomaly'] = iso_forest.fit_predict(df_ts[['y', 'Volume']])
df_ts['anomaly_score'] = iso_forest.score_samples(df_ts[['y', 'Volume']])

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Plot 1: Time series with anomalies highlighted
ax = axes[0]
normal_data = df_ts[df_ts['anomaly'] == 1]
anomaly_data = df_ts[df_ts['anomaly'] == -1]

ax.plot(normal_data['ds'], normal_data['y'], 'b-', linewidth=2, label='Normal', alpha=0.7)
ax.scatter(anomaly_data['ds'], anomaly_data['y'], color='red', s=100, marker='X', 
          label='Anomaly', edgecolors='darkred', linewidth=2, zorder=5)
ax.fill_between(normal_data['ds'], normal_data['y'] - 10, normal_data['y'] + 10, 
               alpha=0.1, color='blue')
ax.set_xlabel('Date', fontsize=11, fontweight='bold')
ax.set_ylabel('Value', fontsize=11, fontweight='bold')
ax.set_title('Time Series Anomaly Detection using Isolation Forest', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 2: Anomaly scores
ax2 = axes[1]
colors = ['red' if x == -1 else 'blue' for x in df_ts['anomaly']]
ax2.bar(range(len(df_ts)), df_ts['anomaly_score'], color=colors, alpha=0.6, edgecolor='black', linewidth=0.5)
ax2.axhline(y=iso_forest.offset_, color='green', linestyle='--', linewidth=2, label='Decision Boundary')
ax2.set_xlabel('Sample Index', fontsize=11, fontweight='bold')
ax2.set_ylabel('Anomaly Score', fontsize=11, fontweight='bold')
ax2.set_title('Anomaly Scores (Lower = More Normal)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

anomaly_count = (df_ts['anomaly'] == -1).sum()
print("✓ Isolation Forest Anomaly Detection Complete")
print(f"  - Detected {anomaly_count} anomalies out of {len(df_ts)} samples")
print(f"  - Anomaly percentage: {100*anomaly_count/len(df_ts):.2f}%")

---

# PREDICTIVE ANALYTICS

## What is Predictive Analytics?
Predictive analytics uses historical data patterns to forecast future outcomes. It answers the question: **"What will happen?"**

---

## 6. Predictive Analytics: Prophet Time Series Forecasting

In [ ]:
# Train test split for fair evaluation
train_size = int(len(df_ts) * 0.8)
df_train = df_ts[['ds', 'y']].iloc[:train_size].copy()
df_test = df_ts[['ds', 'y']].iloc[train_size:].copy()

# Build Prophet model
model_prophet = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
model_prophet.fit(df_train)

# Make future dataframe with test period
future = model_prophet.make_future_dataframe(periods=len(df_test))
forecast = model_prophet.predict(future)

# Extract predictions for test period
prophet_pred = forecast.iloc[train_size:][['ds', 'yhat', 'yhat_lower', 'yhat_upper']].reset_index(drop=True)
prophet_pred['y_actual'] = df_test['y'].values

fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 1, figure=fig, height_ratios=[3, 1])

ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(df_train['ds'], df_train['y'], 'b-', linewidth=2, label='Training Data', alpha=0.8)
ax1.plot(prophet_pred['ds'], prophet_pred['y_actual'], 'g-', linewidth=2, label='Test Data (Actual)', alpha=0.8)
ax1.plot(prophet_pred['ds'], prophet_pred['yhat'], 'r--', linewidth=2, label='Prophet Forecast')
ax1.fill_between(prophet_pred['ds'], prophet_pred['yhat_lower'], prophet_pred['yhat_upper'], 
                alpha=0.2, color='red', label='95% Confidence Interval')
ax1.set_ylabel('Value', fontsize=11, fontweight='bold')
ax1.set_title('Prophet: Actual vs Predicted Values', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10, loc='upper left')
ax1.grid(True, alpha=0.3)

# Residuals
residuals_prophet = prophet_pred['y_actual'] - prophet_pred['yhat']
ax2 = fig.add_subplot(gs[1, 0])
ax2.bar(range(len(residuals_prophet)), residuals_prophet, color=['green' if x > 0 else 'red' for x in residuals_prophet], alpha=0.7, edgecolor='black')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax2.set_xlabel('Time Step', fontsize=11, fontweight='bold')
ax2.set_ylabel('Residual', fontsize=11, fontweight='bold')
ax2.set_title('Prediction Residuals', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle('Prophet Time Series Forecasting Model', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

mae_prophet = mean_absolute_error(prophet_pred['y_actual'], prophet_pred['yhat'])
rmse_prophet = np.sqrt(mean_squared_error(prophet_pred['y_actual'], prophet_pred['yhat']))
mape_prophet = np.mean(np.abs((prophet_pred['y_actual'] - prophet_pred['yhat']) / prophet_pred['y_actual'])) * 100

print("✓ Prophet Model Complete")
print(f"  - MAE: {mae_prophet:.4f}")
print(f"  - RMSE: {rmse_prophet:.4f}")
print(f"  - MAPE: {mape_prophet:.4f}%")

## 7. Predictive Analytics: SARIMA and SARIMAX Models

In [ ]:
# SARIMA Model (Seasonal ARIMA)
# Parameters: (p,d,q) x (P,D,Q,s)
sarima_order = (1, 1, 1)
sarima_seasonal_order = (1, 1, 1, 365)

model_sarima = SARIMAX(df_train['y'], 
                       order=sarima_order, 
                       seasonal_order=sarima_seasonal_order,
                       enforce_stationarity=False,
                       enforce_invertibility=False)
result_sarima = model_sarima.fit(disp=False)

# Make predictions
sarima_predictions = result_sarima.get_forecast(steps=len(df_test)).conf_int()
sarima_pred_mean = result_sarima.get_forecast(steps=len(df_test)).predicted_mean

# SARIMAX - with external variable (Volume)
model_sarimax = SARIMAX(df_train['y'],
                        exog=df_train[['Volume']],
                        order=sarima_order,
                        seasonal_order=sarima_seasonal_order,
                        enforce_stationarity=False,
                        enforce_invertibility=False)
result_sarimax = model_sarimax.fit(disp=False)

sarimax_predictions = result_sarimax.get_forecast(steps=len(df_test), exog=df_test[['Volume']]).conf_int()
sarimax_pred_mean = result_sarimax.get_forecast(steps=len(df_test), exog=df_test[['Volume']]).predicted_mean

fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 2, figure=fig)

# SARIMA
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(df_train['ds'], df_train['y'], 'b-', linewidth=2, label='Training Data', alpha=0.8)
ax1.plot(df_test['ds'], df_test['y'], 'g-', linewidth=2, label='Test Data (Actual)', alpha=0.8)
ax1.plot(df_test['ds'], sarima_pred_mean, 'r--', linewidth=2, label='SARIMA Forecast')
ax1.fill_between(df_test['ds'], 
                sarima_predictions.iloc[:, 0], 
                sarima_predictions.iloc[:, 1],
                alpha=0.2, color='red', label='95% CI')
ax1.set_ylabel('Value', fontsize=11, fontweight='bold')
ax1.set_title('SARIMA: Actual vs Predicted', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# SARIMAX
ax2 = fig.add_subplot(gs[1, :])
ax2.plot(df_train['ds'], df_train['y'], 'b-', linewidth=2, label='Training Data', alpha=0.8)
ax2.plot(df_test['ds'], df_test['y'], 'g-', linewidth=2, label='Test Data (Actual)', alpha=0.8)
ax2.plot(df_test['ds'], sarimax_pred_mean, 'purple', linestyle='--', linewidth=2, label='SARIMAX Forecast (with Volume)')
ax2.fill_between(df_test['ds'],
                sarimax_predictions.iloc[:, 0],
                sarimax_predictions.iloc[:, 1],
                alpha=0.2, color='purple', label='95% CI')
ax2.set_xlabel('Date', fontsize=11, fontweight='bold')
ax2.set_ylabel('Value', fontsize=11, fontweight='bold')
ax2.set_title('SARIMAX: Actual vs Predicted (with External Variable)', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.suptitle('SARIMA vs SARIMAX Models Comparison', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

mae_sarima = mean_absolute_error(df_test['y'], sarima_pred_mean)
rmse_sarima = np.sqrt(mean_squared_error(df_test['y'], sarima_pred_mean))
mae_sarimax = mean_absolute_error(df_test['y'], sarimax_pred_mean)
rmse_sarimax = np.sqrt(mean_squared_error(df_test['y'], sarimax_pred_mean))

print("✓ SARIMA & SARIMAX Models Complete")
print(f"\nSARIMA:")
print(f"  - MAE: {mae_sarima:.4f}")
print(f"  - RMSE: {rmse_sarima:.4f}")
print(f"\nSARIMAX (with external variable):")
print(f"  - MAE: {mae_sarimax:.4f}")
print(f"  - RMSE: {rmse_sarimax:.4f}")

## 8. Predictive Analytics: Exponential Smoothing Methods

In [ ]:
# Simple Exponential Smoothing
model_ses = SimpleExpSmoothing(df_train['y']).fit()
ses_pred = model_ses.forecast(steps=len(df_test))

# Holt's Linear Trend
model_holt = Holt(df_train['y']).fit()
holt_pred = model_holt.forecast(steps=len(df_test))

# Holt-Winters Exponential Smoothing (with seasonality)
model_hw = ExponentialSmoothing(df_train['y'], seasonal_periods=365, trend='add', seasonal='add').fit()
hw_pred = model_hw.forecast(steps=len(df_test))

fig = plt.figure(figsize=(16, 12))
gs = GridSpec(3, 1, figure=fig)

# Simple Exponential Smoothing
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(df_train['ds'], df_train['y'], 'b-', linewidth=2, label='Training Data', alpha=0.8)
ax1.plot(df_test['ds'], df_test['y'], 'g-', linewidth=2, label='Test Data (Actual)', alpha=0.8)
ax1.plot(df_test['ds'], ses_pred, 'orange', linestyle='--', linewidth=2, label='SES Forecast')
ax1.set_ylabel('Value', fontsize=11, fontweight='bold')
ax1.set_title('Simple Exponential Smoothing: Actual vs Predicted', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Holt's Linear Trend
ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(df_train['ds'], df_train['y'], 'b-', linewidth=2, label='Training Data', alpha=0.8)
ax2.plot(df_test['ds'], df_test['y'], 'g-', linewidth=2, label='Test Data (Actual)', alpha=0.8)
ax2.plot(df_test['ds'], holt_pred, 'cyan', linestyle='--', linewidth=2, label="Holt's Linear Trend")
ax2.set_ylabel('Value', fontsize=11, fontweight='bold')
ax2.set_title("Holt's Linear Trend: Actual vs Predicted", fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# Holt-Winters
ax3 = fig.add_subplot(gs[2, 0])
ax3.plot(df_train['ds'], df_train['y'], 'b-', linewidth=2, label='Training Data', alpha=0.8)
ax3.plot(df_test['ds'], df_test['y'], 'g-', linewidth=2, label='Test Data (Actual)', alpha=0.8)
ax3.plot(df_test['ds'], hw_pred, 'brown', linestyle='--', linewidth=2, label='Holt-Winters Forecast')
ax3.set_xlabel('Date', fontsize=11, fontweight='bold')
ax3.set_ylabel('Value', fontsize=11, fontweight='bold')
ax3.set_title('Holt-Winters Exponential Smoothing: Actual vs Predicted', fontsize=13, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

plt.suptitle('Exponential Smoothing Methods Comparison', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

mae_ses = mean_absolute_error(df_test['y'], ses_pred)
mae_holt = mean_absolute_error(df_test['y'], holt_pred)
mae_hw = mean_absolute_error(df_test['y'], hw_pred)

print("✓ Exponential Smoothing Models Complete")
print(f"\nSimple Exponential Smoothing (SES):")
print(f"  - MAE: {mae_ses:.4f}")
print(f"\nHolt's Linear Trend:")
print(f"  - MAE: {mae_holt:.4f}")
print(f"\nHolt-Winters (with seasonality):")
print(f"  - MAE: {mae_hw:.4f}")

## 9. Predictive Analytics: Logistic Regression for Churn Prediction

In [ ]:
# Prepare data for churn model
X_churn = customer_data[['Spending', 'Visits', 'Days_Since_Purchase', 'Age', 'Account_Age']].copy()
y_churn = customer_data['Churn'].copy()

X_scaled = StandardScaler().fit_transform(X_churn)
X_train_ch, X_test_ch, y_train_ch, y_test_ch = train_test_split(X_scaled, y_churn, test_size=0.3, random_state=42)

# Logistic Regression Model
model_lr = LogisticRegression(random_state=42, max_iter=1000)
model_lr.fit(X_train_ch, y_train_ch)

y_pred_lr = model_lr.predict(X_test_ch)
y_pred_proba = model_lr.predict_proba(X_test_ch)[:, 1]

# Confusion Matrix
cm = confusion_matrix(y_test_ch, y_pred_lr)

fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 2, figure=fig)

# Confusion Matrix
ax1 = fig.add_subplot(gs[0, 0])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1, cbar=True, 
            xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
ax1.set_xlabel('Predicted', fontsize=11, fontweight='bold')
ax1.set_ylabel('Actual', fontsize=11, fontweight='bold')
ax1.set_title('Confusion Matrix: Churn Prediction', fontsize=12, fontweight='bold')

# ROC Curve
from sklearn.metrics import roc_curve, auc
fpr, tpr, thresholds = roc_curve(y_test_ch, y_pred_proba)
roc_auc = auc(fpr, tpr)

ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
ax2.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
ax2.fill_between(fpr, tpr, alpha=0.2, color='darkorange')
ax2.set_xlabel('False Positive Rate', fontsize=11, fontweight='bold')
ax2.set_ylabel('True Positive Rate', fontsize=11, fontweight='bold')
ax2.set_title('ROC Curve', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# Feature Importance (Coefficients)
feature_names = ['Spending', 'Visits', 'Days_Since_Purchase', 'Age', 'Account_Age']
ax3 = fig.add_subplot(gs[1, 0])
colors_coef = ['green' if x > 0 else 'red' for x in model_lr.coef_[0]]
ax3.barh(feature_names, model_lr.coef_[0], color=colors_coef, alpha=0.7, edgecolor='black')
ax3.set_xlabel('Coefficient Value', fontsize=11, fontweight='bold')
ax3.set_title('Feature Importance (Logistic Regression Coefficients)', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='x')

# Predicted Probabilities Distribution
ax4 = fig.add_subplot(gs[1, 1])
ax4.hist(y_pred_proba[y_test_ch == 0], bins=30, alpha=0.6, label='No Churn', color='blue', edgecolor='black')
ax4.hist(y_pred_proba[y_test_ch == 1], bins=30, alpha=0.6, label='Churn', color='red', edgecolor='black')
ax4.axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Decision Threshold')
ax4.set_xlabel('Predicted Probability', fontsize=11, fontweight='bold')
ax4.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax4.set_title('Distribution of Predicted Probabilities', fontsize=12, fontweight='bold')
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3, axis='y')

plt.suptitle('Logistic Regression: Churn Prediction Model', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

accuracy = (y_pred_lr == y_test_ch).mean()
precision = cm[1, 1] / (cm[0, 1] + cm[1, 1])
recall = cm[1, 1] / (cm[1, 0] + cm[1, 1])
f1 = 2 * (precision * recall) / (precision + recall)

print("✓ Logistic Regression Churn Model Complete")
print(f"  - Accuracy: {accuracy:.4f}")
print(f"  - Precision: {precision:.4f}")
print(f"  - Recall: {recall:.4f}")
print(f"  - F1-Score: {f1:.4f}")
print(f"  - ROC-AUC: {roc_auc:.4f}")

## 10. Predictive Analytics: XGBoost with SHAP Explainability

In [ ]:
# Train XGBoost model
model_xgb = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
model_xgb.fit(X_train_ch, y_train_ch)

y_pred_xgb = model_xgb.predict(X_test_ch)
y_pred_proba_xgb = model_xgb.predict_proba(X_test_ch)[:, 1]

# SHAP values for explainability
explainer = shap.TreeExplainer(model_xgb)
shap_values = explainer.shap_values(X_test_ch)

# Gradient Boosting for comparison
model_gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
model_gb.fit(X_train_ch, y_train_ch)
y_pred_gb = model_gb.predict(X_test_ch)

fig = plt.figure(figsize=(16, 12))
gs = GridSpec(3, 2, figure=fig)

# Feature Importance (XGBoost)
ax1 = fig.add_subplot(gs[0, 0])
importance = model_xgb.feature_importances_
ax1.barh(feature_names, importance, color='steelblue', alpha=0.8, edgecolor='black')
ax1.set_xlabel('Importance', fontsize=11, fontweight='bold')
ax1.set_title('XGBoost Feature Importance', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')

# Feature Importance (Gradient Boosting)
ax2 = fig.add_subplot(gs[0, 1])
importance_gb = model_gb.feature_importances_
ax2.barh(feature_names, importance_gb, color='coral', alpha=0.8, edgecolor='black')
ax2.set_xlabel('Importance', fontsize=11, fontweight='bold')
ax2.set_title('Gradient Boosting Feature Importance', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

# Model Comparison - Confusion Matrices
ax3 = fig.add_subplot(gs[1, 0])
cm_xgb = confusion_matrix(y_test_ch, y_pred_xgb)
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='YlOrRd', ax=ax3, cbar=True,
            xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
ax3.set_xlabel('Predicted', fontsize=11, fontweight='bold')
ax3.set_ylabel('Actual', fontsize=11, fontweight='bold')
ax3.set_title('XGBoost Confusion Matrix', fontsize=12, fontweight='bold')

ax4 = fig.add_subplot(gs[1, 1])
cm_gb = confusion_matrix(y_test_ch, y_pred_gb)
sns.heatmap(cm_gb, annot=True, fmt='d', cmap='PuBuGn', ax=ax4, cbar=True,
            xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
ax4.set_xlabel('Predicted', fontsize=11, fontweight='bold')
ax4.set_ylabel('Actual', fontsize=11, fontweight='bold')
ax4.set_title('Gradient Boosting Confusion Matrix', fontsize=12, fontweight='bold')

# SHAP Summary Plot (Mean |SHAP| values)
ax5 = fig.add_subplot(gs[2, :])
mean_shap_values = np.abs(shap_values).mean(axis=0)
indices = np.argsort(mean_shap_values)
ax5.barh(np.array(feature_names)[indices], mean_shap_values[indices], color='darkgreen', alpha=0.8, edgecolor='black')
ax5.set_xlabel('Mean |SHAP Value|', fontsize=11, fontweight='bold')
ax5.set_title('SHAP Feature Importance (Mean Absolute Values)', fontsize=12, fontweight='bold')
ax5.grid(True, alpha=0.3, axis='x')

plt.suptitle('XGBoost vs Gradient Boosting with SHAP Explainability', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

accuracy_xgb = (y_pred_xgb == y_test_ch).mean()
accuracy_gb = (y_pred_gb == y_test_ch).mean()

print("✓ XGBoost and Gradient Boosting Models Complete")
print(f"\nXGBoost Accuracy: {accuracy_xgb:.4f}")
print(f"Gradient Boosting Accuracy: {accuracy_gb:.4f}")
print(f"\nSHAP Analysis:")
print(f"  - Most important feature: {feature_names[np.argmax(mean_shap_values)]}")
print(f"  - Mean |SHAP| value: {np.max(mean_shap_values):.4f}")

## 11. Predictive Analytics: Markov Chain State Analysis

In [ ]:
# Define customer states based on spending pattern
customer_data['State'] = pd.cut(customer_data['Spending'], bins=[0, 50, 100, 150, 200, np.inf],
                               labels=['Low', 'Medium', 'High', 'Very High', 'Premium'])

# Create transition matrix - count transitions between states
states = customer_data['State'].unique()
transition_counts = pd.DataFrame(0, index=states, columns=states)

# Simulate state transitions (simplified: based on random walk)
np.random.seed(42)
n_transitions = 10000
current_state_idx = np.random.randint(0, len(states))

for _ in range(n_transitions):
    current_state = states[current_state_idx]
    # Probability of staying in same state or moving to adjacent states
    transition_probs = np.array([0.6 if i == current_state_idx else 0.1 for i in range(len(states))])
    transition_probs /= transition_probs.sum()
    
    next_state_idx = np.random.choice(len(states), p=transition_probs)
    transition_counts.loc[current_state, states[next_state_idx]] += 1
    current_state_idx = next_state_idx

# Convert to probabilities
transition_matrix = transition_counts.div(transition_counts.sum(axis=1), axis=0)

# Calculate steady-state probabilities using eigenvalues
eigenvalues, eigenvectors = np.linalg.eig(transition_matrix.T.values)
steady_state_idx = np.argmax(np.abs(eigenvalues - 1) < 1e-8)
steady_state = np.abs(eigenvectors[:, steady_state_idx]) / np.abs(eigenvectors[:, steady_state_idx]).sum()

fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 2, figure=fig)

# Transition Matrix Heatmap
ax1 = fig.add_subplot(gs[0, :])
sns.heatmap(transition_matrix, annot=True, fmt='.3f', cmap='RdYlGn', ax=ax1, cbar_kws={'label': 'Transition Probability'},
            xticklabels=states, yticklabels=states)
ax1.set_xlabel('To State', fontsize=11, fontweight='bold')
ax1.set_ylabel('From State', fontsize=11, fontweight='bold')
ax1.set_title('Markov Chain Transition Matrix', fontsize=13, fontweight='bold')

# State Distribution Over Time
ax2 = fig.add_subplot(gs[1, 0])
state_initial = np.array([0.2, 0.3, 0.3, 0.15, 0.05])  # Initial distribution
state_dist = state_initial.copy()
steps = 50
states_over_time = [state_dist.copy()]

for _ in range(steps):
    state_dist = state_dist @ transition_matrix.values
    states_over_time.append(state_dist.copy())

states_over_time = np.array(states_over_time)
for i, state in enumerate(states):
    ax2.plot(states_over_time[:, i], marker='o', label=state, linewidth=2)
ax2.set_xlabel('Time Step', fontsize=11, fontweight='bold')
ax2.set_ylabel('Probability', fontsize=11, fontweight='bold')
ax2.set_title('State Distribution Convergence', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# Steady-State Probabilities
ax3 = fig.add_subplot(gs[1, 1])
bars = ax3.bar(states, steady_state, color=['red', 'orange', 'yellow', 'lightgreen', 'darkgreen'], 
              edgecolor='black', linewidth=2, alpha=0.8)
ax3.set_ylabel('Probability', fontsize=11, fontweight='bold')
ax3.set_title('Steady-State Distribution', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, val in zip(bars, steady_state):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold')

plt.suptitle('Markov Chain Analysis: Customer State Transitions', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("✓ Markov Chain Analysis Complete")
print(f"\nSteady-State Probabilities:")
for state, prob in zip(states, steady_state):
    print(f"  {state}: {prob:.4f}")

---

# PRESCRIPTIVE ANALYTICS

## What is Prescriptive Analytics?
Prescriptive analytics uses optimization and simulation to recommend actions. It answers the question: **"What should we do?"**

---

## 12. Prescriptive Analytics: Linear Programming

In [ ]:
# Example: Production Optimization Problem
# A company produces 2 products with limited resources
# Product A: profit=$40, resource1=2 units, resource2=1 unit
# Product B: profit=$30, resource1=1 unit, resource2=2 units
# Available: 100 units of resource1, 100 units of resource2

try:
    from pulp import *
    
    # Create the LP problem
    prob = LpProblem("Production_Optimization", LpMaximize)
    
    # Decision variables
    x_a = LpVariable("Product_A", lowBound=0, cat='Continuous')
    x_b = LpVariable("Product_B", lowBound=0, cat='Continuous')
    
    # Objective function: Maximize profit
    prob += 40 * x_a + 30 * x_b, "Total_Profit"
    
    # Constraints
    prob += 2 * x_a + 1 * x_b <= 100, "Resource_1_Limit"
    prob += 1 * x_a + 2 * x_b <= 100, "Resource_2_Limit"
    
    # Solve
    prob.solve(PULP_CBC_CMD(msg=0))
    
    lp_solved = True
    lp_status = LpStatus[prob.status]
    lp_solution = {
        'Product_A': value(x_a),
        'Product_B': value(x_b),
        'Max_Profit': value(prob.objective)
    }
except:
    lp_solved = False
    print("Note: PuLP not available. Using alternative formulation.")

# Create visualization for LP
fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 2, figure=fig)

# Feasible region visualization
ax1 = fig.add_subplot(gs[0, 0])
x = np.linspace(0, 60, 200)

# Constraint lines
y1 = (100 - 2*x) / 1  # Resource 1: 2x + y <= 100
y2 = (100 - x) / 2    # Resource 2: x + 2y <= 100

ax1.plot(x, y1, 'r-', linewidth=2, label='Resource 1: 2A + B ≤ 100')
ax1.plot(x, y2, 'b-', linewidth=2, label='Resource 2: A + 2B ≤ 100')
ax1.fill_between(x, 0, np.minimum(y1, y2), where=(np.minimum(y1, y2) >= 0), 
                 alpha=0.3, color='green', label='Feasible Region')

if lp_solved:
    ax1.plot(lp_solution['Product_A'], lp_solution['Product_B'], 'r*', markersize=20, 
            label=f"Optimal: A={lp_solution['Product_A']:.1f}, B={lp_solution['Product_B']:.1f}", zorder=5)

ax1.set_xlim(0, 60)
ax1.set_ylim(0, 60)
ax1.set_xlabel('Product A (units)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Product B (units)', fontsize=11, fontweight='bold')
ax1.set_title('Linear Programming: Feasible Region & Optimal Solution', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Profit function contours
ax2 = fig.add_subplot(gs[0, 1])
X, Y = np.meshgrid(np.linspace(0, 60, 100), np.linspace(0, 60, 100))
Z = 40*X + 30*Y  # Profit function

contour = ax2.contourf(X, Y, Z, levels=20, cmap='viridis', alpha=0.8)
contour_lines = ax2.contour(X, Y, Z, levels=10, colors='black', alpha=0.3, linewidths=0.5)
ax2.clabel(contour_lines, inline=True, fontsize=8)

if lp_solved:
    ax2.plot(lp_solution['Product_A'], lp_solution['Product_B'], 'r*', markersize=20, 
            label=f"Optimal Point\nProfit: ${lp_solution['Max_Profit']:.2f}", zorder=5)

ax2.set_xlabel('Product A (units)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Product B (units)', fontsize=11, fontweight='bold')
ax2.set_title('Profit Contours (Objective Function)', fontsize=12, fontweight='bold')
plt.colorbar(contour, ax=ax2, label='Profit ($)')
ax2.legend(fontsize=10)

# Sensitivity Analysis
if lp_solved:
    ax3 = fig.add_subplot(gs[1, :])
    
    # Vary profit of Product A
    profit_a_range = np.linspace(20, 60, 50)
    optimal_a = []
    optimal_b = []
    total_profit = []
    
    for pa in profit_a_range:
        try:
            prob2 = LpProblem("Sensitivity", LpMaximize)
            xa = LpVariable("A", lowBound=0, cat='Continuous')
            xb = LpVariable("B", lowBound=0, cat='Continuous')
            prob2 += pa * xa + 30 * xb, "Profit"
            prob2 += 2*xa + xb <= 100, "R1"
            prob2 += xa + 2*xb <= 100, "R2"
            prob2.solve(PULP_CBC_CMD(msg=0))
            
            optimal_a.append(value(xa))
            optimal_b.append(value(xb))
            total_profit.append(value(prob2.objective))
        except:
            pass
    
    ax3.plot(profit_a_range[:len(total_profit)], total_profit, 'g-o', linewidth=2, markersize=6, label='Max Profit')
    ax3.axvline(x=40, color='red', linestyle='--', linewidth=2, label='Original Profit A ($40)')
    ax3.set_xlabel('Product A Profit ($)', fontsize=11, fontweight='bold')
    ax3.set_ylabel('Total Profit ($)', fontsize=11, fontweight='bold')
    ax3.set_title('Sensitivity Analysis: Impact of Product A Price on Optimal Profit', fontsize=12, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    ax3.legend(fontsize=10)

plt.suptitle('Linear Programming: Production Optimization', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

if lp_solved:
    print("✓ Linear Programming Optimization Complete")
    print(f"  - Status: {lp_status}")
    print(f"  - Optimal Product A: {lp_solution['Product_A']:.2f} units")
    print(f"  - Optimal Product B: {lp_solution['Product_B']:.2f} units")
    print(f"  - Maximum Profit: ${lp_solution['Max_Profit']:.2f}")
else:
    print("✓ Linear Programming formulation shown (solver not available for execution)")

## 13. Prescriptive Analytics: Mixed-Integer Linear Programming (MILP)

In [ ]:
# Facility Location Problem: Decide which warehouses to open
# 3 potential warehouse locations, 4 customer zones
# Binary decision: open warehouse or not
# Continuous decision: quantity shipped from each warehouse to each customer

try:
    from pulp import *
    
    # Data
    warehouses = ['W1', 'W2', 'W3']
    customers = ['C1', 'C2', 'C3', 'C4']
    fixed_cost = {'W1': 1000, 'W2': 1200, 'W3': 1100}  # Fixed cost to open
    shipping_cost = {
        'W1': [10, 12, 14, 11],
        'W2': [13, 11, 10, 12],
        'W3': [12, 13, 11, 10]
    }
    demand = [100, 150, 120, 130]
    capacity_if_open = 200  # Max supply per warehouse if opened
    
    # Create MILP
    prob_milp = LpProblem("Facility_Location", LpMinimize)
    
    # Decision variables
    y = {w: LpVariable(f"Open_{w}", cat='Binary') for w in warehouses}  # Binary
    x = {(w, c): LpVariable(f"Ship_{w}_to_{c}", lowBound=0, cat='Continuous')
         for w in warehouses for c in customers}
    
    # Objective: Minimize fixed cost + shipping cost
    prob_milp += (lpSum([fixed_cost[w] * y[w] for w in warehouses]) +
                 lpSum([shipping_cost[warehouses.index(w)][customers.index(c)] * x[(w, c)]
                        for w in warehouses for c in customers]))
    
    # Constraints
    for i, c in enumerate(customers):
        prob_milp += lpSum([x[(w, c)] for w in warehouses]) == demand[i], f"Demand_{c}"
    
    for w in warehouses:
        prob_milp += lpSum([x[(w, c)] for c in customers]) <= capacity_if_open * y[w], f"Capacity_{w}"
    
    # Solve
    prob_milp.solve(PULP_CBC_CMD(msg=0))
    
    milp_solved = True
    milp_status = LpStatus[prob_milp.status]
    opened_facilities = [w for w in warehouses if y[w].varValue == 1]
    milp_cost = value(prob_milp.objective)
except:
    milp_solved = False
    print("Note: MILP solver not fully available.")

# Visualization
fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 2, figure=fig)

# Heuristic visualization (showing concept even if solver unavailable)
# Simulate opening closest 2 warehouses
ax1 = fig.add_subplot(gs[0, 0])
opened = ['W1', 'W2']  # If solver not available, use heuristic
ax1.barh(warehouses, [1 if w in opened else 0 for w in warehouses], 
        color=['green' if w in opened else 'red' for w in warehouses], alpha=0.7, edgecolor='black', linewidth=2)
ax1.set_xlabel('Opened (1) or Closed (0)', fontsize=11, fontweight='bold')
ax1.set_title('Facility Location Decisions', fontsize=12, fontweight='bold')
ax1.set_xlim(0, 1.2)

# Fixed costs
ax2 = fig.add_subplot(gs[0, 1])
costs_values = [fixed_cost[w] for w in warehouses]
colors_cost = ['green' if w in opened else 'gray' for w in warehouses]
ax2.bar(warehouses, costs_values, color=colors_cost, alpha=0.7, edgecolor='black', linewidth=2)
ax2.set_ylabel('Fixed Cost ($)', fontsize=11, fontweight='bold')
ax2.set_title('Facility Fixed Costs', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# Shipping routes visualization
ax3 = fig.add_subplot(gs[1, :])
if milp_solved:
    # Create network visualization
    shipping_summary = {}
    for (w, c), amount in x.items():
        if amount.varValue > 0:
            shipping_summary[f"{w}→{c}"] = amount.varValue
else:
    # Heuristic: distribute from opened warehouses
    shipping_summary = {
        'W1→C1': 80, 'W1→C2': 70,
        'W2→C2': 80, 'W2→C3': 120, 'W2→C4': 10,
    }

routes = list(shipping_summary.keys())
amounts = list(shipping_summary.values())
colors_route = plt.cm.Set3(np.linspace(0, 1, len(routes)))

bars = ax3.barh(routes, amounts, color=colors_route, edgecolor='black', linewidth=1.5, alpha=0.8)
ax3.set_xlabel('Quantity Shipped (units)', fontsize=11, fontweight='bold')
ax3.set_title('Optimal Shipping Routes & Quantities', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='x')

# Add value labels
for bar, val in zip(bars, amounts):
    width = bar.get_width()
    ax3.text(width, bar.get_y() + bar.get_height()/2., f'{val:.0f}',
            ha='left', va='center', fontweight='bold', fontsize=10)

plt.suptitle('MILP: Facility Location & Network Design', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

if milp_solved:
    print("✓ MILP (Mixed-Integer Linear Programming) Complete")
    print(f"  - Status: {milp_status}")
    print(f"  - Opened Facilities: {opened_facilities}")
    print(f"  - Total Cost: ${milp_cost:.2f}")
else:
    print("✓ MILP Model formulation demonstrated (binary variables for facility opening)")

## 14. Prescriptive Analytics: Monte Carlo Simulation

In [ ]:
# Monte Carlo Simulation for Project Cost & Time Estimation
# Estimate: Completion time and total cost with uncertainty

np.random.seed(42)
n_simulations = 10000

# Parameters: (mean, std_dev)
task_durations = {
    'Task 1': (10, 2),
    'Task 2': (15, 3),
    'Task 3': (12, 2.5),
    'Task 4': (8, 1.5)
}

costs_per_task = {
    'Task 1': (5000, 500),
    'Task 2': (8000, 1000),
    'Task 3': (6000, 800),
    'Task 4': (4000, 400)
}

# Run simulations
total_duration = np.zeros(n_simulations)
total_cost = np.zeros(n_simulations)

for i in range(n_simulations):
    duration = 0
    cost = 0
    for task in task_durations:
        mean_d, std_d = task_durations[task]
        mean_c, std_c = costs_per_task[task]
        
        duration += np.random.normal(mean_d, std_d)
        cost += np.random.normal(mean_c, std_c)
    
    total_duration[i] = max(0, duration)  # No negative durations
    total_cost[i] = max(0, cost)  # No negative costs

# Calculate statistics
duration_stats = {
    'Mean': np.mean(total_duration),
    'Std': np.std(total_duration),
    'P10': np.percentile(total_duration, 10),
    'P50': np.percentile(total_duration, 50),
    'P90': np.percentile(total_duration, 90)
}

cost_stats = {
    'Mean': np.mean(total_cost),
    'Std': np.std(total_cost),
    'P10': np.percentile(total_cost, 10),
    'P50': np.percentile(total_cost, 50),
    'P90': np.percentile(total_cost, 90)
}

fig = plt.figure(figsize=(16, 12))
gs = GridSpec(3, 2, figure=fig)

# Duration distribution
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(total_duration, bins=50, density=True, alpha=0.7, color='skyblue', edgecolor='black')
ax1.axvline(duration_stats['Mean'], color='red', linestyle='--', linewidth=2, label='Mean')
ax1.axvline(duration_stats['P10'], color='orange', linestyle=':', linewidth=2, label='P10')
ax1.axvline(duration_stats['P90'], color='green', linestyle=':', linewidth=2, label='P90')
ax1.set_xlabel('Project Duration (days)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Probability Density', fontsize=11, fontweight='bold')
ax1.set_title('Project Duration Distribution (Monte Carlo)', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Cost distribution
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(total_cost, bins=50, density=True, alpha=0.7, color='lightcoral', edgecolor='black')
ax2.axvline(cost_stats['Mean'], color='red', linestyle='--', linewidth=2, label='Mean')
ax2.axvline(cost_stats['P10'], color='orange', linestyle=':', linewidth=2, label='P10')
ax2.axvline(cost_stats['P90'], color='green', linestyle=':', linewidth=2, label='P90')
ax2.set_xlabel('Project Cost ($)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Probability Density', fontsize=11, fontweight='bold')
ax2.set_title('Project Cost Distribution (Monte Carlo)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# Cumulative distributions
ax3 = fig.add_subplot(gs[1, 0])
sorted_duration = np.sort(total_duration)
cumulative = np.arange(1, len(sorted_duration) + 1) / len(sorted_duration)
ax3.plot(sorted_duration, cumulative, linewidth=2, color='darkblue')
ax3.axvline(duration_stats['P50'], color='red', linestyle='--', linewidth=2, label='Median (P50)')
ax3.axhline(0.9, color='green', linestyle=':', linewidth=1, alpha=0.5)
ax3.scatter([duration_stats['P90']], [0.9], color='green', s=100, zorder=5, label=f"P90: {duration_stats['P90']:.1f} days")
ax3.set_xlabel('Project Duration (days)', fontsize=11, fontweight='bold')
ax3.set_ylabel('Cumulative Probability', fontsize=11, fontweight='bold')
ax3.set_title('Duration CDF: Risk Analysis', fontsize=12, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

# Cost CDF
ax4 = fig.add_subplot(gs[1, 1])
sorted_cost = np.sort(total_cost)
cumulative_cost = np.arange(1, len(sorted_cost) + 1) / len(sorted_cost)
ax4.plot(sorted_cost, cumulative_cost, linewidth=2, color='darkred')
ax4.axvline(cost_stats['P50'], color='red', linestyle='--', linewidth=2, label='Median (P50)')
ax4.axhline(0.9, color='green', linestyle=':', linewidth=1, alpha=0.5)
ax4.scatter([cost_stats['P90']], [0.9], color='green', s=100, zorder=5, label=f"P90: ${cost_stats['P90']:,.0f}")
ax4.set_xlabel('Project Cost ($)', fontsize=11, fontweight='bold')
ax4.set_ylabel('Cumulative Probability', fontsize=11, fontweight='bold')
ax4.set_title('Cost CDF: Risk Analysis', fontsize=12, fontweight='bold')
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3)

# Statistics table
ax5 = fig.add_subplot(gs[2, :])
ax5.axis('tight')
ax5.axis('off')

stats_data = []
stats_data.append(['Metric', 'Duration (days)', 'Cost ($)'])
for metric in ['Mean', 'Std', 'P10', 'P50', 'P90']:
    stats_data.append([metric, f"{duration_stats[metric]:.2f}", f"${cost_stats[metric]:,.0f}"])

# Add risk assessment
risk_duration_exceed_50 = np.mean(total_duration > 50) * 100
risk_cost_exceed_35k = np.mean(total_cost > 35000) * 100
stats_data.append(['P(Duration>50d)', f"{risk_duration_exceed_50:.1f}%", '---'])
stats_data.append(['P(Cost>$35k)', '---', f"{risk_cost_exceed_35k:.1f}%"])

table = ax5.table(cellText=stats_data, cellLoc='center', loc='center',
                 colWidths=[0.3, 0.35, 0.35])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)

# Style header row
for i in range(3):
    table[(0, i)].set_facecolor('#4CAF50')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Alternate row colors
for i in range(1, len(stats_data)):
    for j in range(3):
        if i % 2 == 0:
            table[(i, j)].set_facecolor('#f0f0f0')
        else:
            table[(i, j)].set_facecolor('white')

ax5.set_title('Monte Carlo Statistics & Risk Assessment', fontsize=12, fontweight='bold', pad=20)

plt.suptitle('Monte Carlo Simulation: Project Risk Analysis', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("✓ Monte Carlo Simulation Complete")
print(f"\nDuration Estimates:")
print(f"  - Average: {duration_stats['Mean']:.2f} days")
print(f"  - 90% confidence range: {duration_stats['P10']:.2f} - {duration_stats['P90']:.2f} days")
print(f"\nCost Estimates:")
print(f"  - Average: ${cost_stats['Mean']:,.2f}")
print(f"  - 90% confidence range: ${cost_stats['P10']:,.2f} - ${cost_stats['P90']:,.2f}")

## 15. Prescriptive Analytics: Sensitivity Analysis

In [ ]:
# Sensitivity Analysis: Financial Model
# Model: NPV = -Initial_Investment + (Annual_Revenue - Annual_Cost) * (1 - (1+Discount) ^ -Years) / Discount
# Analyze how NPV changes with different parameters

# Base case parameters
base_params = {
    'Initial_Investment': 100000,
    'Annual_Revenue': 50000,
    'Annual_Cost': 15000,
    'Years': 5,
    'Discount_Rate': 0.10
}

def calculate_npv(initial_inv, annual_revenue, annual_cost, years, discount_rate):
    if discount_rate == 0:
        return -initial_inv + (annual_revenue - annual_cost) * years
    pv_factor = (1 - (1 + discount_rate) ** (-years)) / discount_rate
    return -initial_inv + (annual_revenue - annual_cost) * pv_factor

# Base case NPV
base_npv = calculate_npv(**base_params)

# One-way sensitivity (spider diagram)
parameters_to_vary = list(base_params.keys())
sensitivity_results = {}

for param in parameters_to_vary:
    npv_values = []
    variation_values = np.linspace(base_params[param] * 0.5, base_params[param] * 1.5, 20)
    
    for var in variation_values:
        params_temp = base_params.copy()
        params_temp[param] = var
        npv_values.append(calculate_npv(**params_temp))
    
    sensitivity_results[param] = {
        'variations': variation_values,
        'npv_values': npv_values
    }

# Tornado chart
ax1_range = []
ax1_low_npv = []
ax1_high_npv = []
ax1_labels = []

for param in parameters_to_vary:
    npv_vals = sensitivity_results[param]['npv_values']
    ax1_low_npv.append(min(npv_vals))
    ax1_high_npv.append(max(npv_vals))
    ax1_range.append(max(npv_vals) - min(npv_vals))
    ax1_labels.append(param.replace('_', ' '))

# Sort by range (tornado diagram)
sorted_indices = np.argsort(ax1_range)[::-1]
sorted_labels = [ax1_labels[i] for i in sorted_indices]
sorted_low = [ax1_low_npv[i] for i in sorted_indices]
sorted_high = [ax1_high_npv[i] for i in sorted_indices]
sorted_range = [ax1_range[i] for i in sorted_indices]

fig = plt.figure(figsize=(16, 12))
gs = GridSpec(2, 3, figure=fig)

# Tornado Chart
ax1 = fig.add_subplot(gs[0, :])
y_pos = np.arange(len(sorted_labels))

# Plot base NPV line
ax1.axvline(x=base_npv, color='black', linestyle='-', linewidth=2, label=f'Base NPV: ${base_npv:,.0f}')

# Plot bars
for i, label in enumerate(sorted_labels):
    ax1.barh(i, sorted_high[i] - base_npv, left=base_npv, height=0.4, 
            color='green', alpha=0.7, edgecolor='black', linewidth=1)
    ax1.barh(i, sorted_low[i] - base_npv, left=base_npv, height=0.4, 
            color='red', alpha=0.7, edgecolor='black', linewidth=1)

ax1.set_yticks(y_pos)
ax1.set_yticklabels(sorted_labels, fontsize=11, fontweight='bold')
ax1.set_xlabel('NPV Impact ($)', fontsize=11, fontweight='bold')
ax1.set_title('Tornado Diagram: Parameter Sensitivity', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')
ax1.legend(fontsize=10)

# Spider plots for individual parameters
colors_spider = plt.cm.Set2(np.linspace(0, 1, 3))

params_to_plot = ['Annual_Revenue', 'Annual_Cost', 'Discount_Rate']
for idx, param in enumerate(params_to_plot):
    ax = fig.add_subplot(gs[1, idx])
    
    vars_norm = sensitivity_results[param]['variations'] / base_params[param] * 100  # Percentage of base
    npv_vals = sensitivity_results[param]['npv_values']
    
    ax.plot(vars_norm, npv_vals, 'o-', linewidth=2.5, markersize=8, 
           color=colors_spider[idx], markerfacecolor='white', markeredgewidth=2)
    ax.axvline(x=100, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.axhline(y=base_npv, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.scatter([100], [base_npv], color='red', s=200, marker='X', zorder=5, edgecolors='darkred', linewidth=2)
    
    ax.set_xlabel(f'{param.replace(\"_\", \" \")} (% of Base)', fontsize=11, fontweight='bold')
    ax.set_ylabel('NPV ($)', fontsize=11, fontweight='bold')
    ax.set_title(f'Sensitivity to {param.replace(\"_\", \" \")}', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.ticklabel_format(style='plain', axis='y')

plt.suptitle('Sensitivity Analysis: Financial Model (NPV)', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

# Create sensitivity table
elasticity = {}
for param in parameters_to_vary:
    base_val = base_params[param]
    # Calculate elasticity: % change in NPV / % change in parameter
    variation = sensitivity_results[param]['variations']
    npv_vals = sensitivity_results[param]['npv_values']
    
    pct_change_param = (variation - base_val) / base_val
    pct_change_npv = (np.array(npv_vals) - base_npv) / base_npv
    
    # Avoid division by zero
    elasticity[param] = np.mean(np.where(pct_change_param != 0, pct_change_npv / pct_change_param, 0))

print("✓ Sensitivity Analysis Complete")
print(f"\nBase Case NPV: ${base_npv:,.2f}")
print(f"\nElasticity (% change in NPV / % change in parameter):")
for param in sorted(elasticity.keys(), key=lambda x: abs(elasticity[x]), reverse=True):
    print(f"  {param}: {elasticity[param]:.3f}")

## 16. Prescriptive Analytics: Data Envelopment Analysis (DEA)

In [ ]:
# Data Envelopment Analysis: Evaluate efficiency of retail branches
# Inputs: Staff Count, Operating Cost
# Outputs: Sales Revenue, Customer Satisfaction Score

data_dea = pd.DataFrame({
    'Branch': ['Branch A', 'Branch B', 'Branch C', 'Branch D', 'Branch E', 'Branch F'],
    'Staff': [20, 25, 18, 22, 30, 19],
    'Operating_Cost': [50000, 60000, 45000, 55000, 70000, 48000],
    'Revenue': [250000, 280000, 220000, 260000, 290000, 240000],
    'Satisfaction': [4.2, 4.5, 4.0, 4.3, 4.6, 4.1]
})

# Calculate simple efficiency metrics
data_dea['Revenue_per_Staff'] = data_dea['Revenue'] / data_dea['Staff']
data_dea['Revenue_per_Dollar'] = data_dea['Revenue'] / data_dea['Operating_Cost']
data_dea['Efficiency_Score'] = (data_dea['Revenue_per_Staff'] + 
                                 data_dea['Revenue_per_Dollar'] * 1000) / 2  # Scaled

# Normalize efficiency score to 0-1
data_dea['Efficiency_Normalized'] = data_dea['Efficiency_Score'] / data_dea['Efficiency_Score'].max()

fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 2, figure=fig)

# Efficiency Frontier
ax1 = fig.add_subplot(gs[0, 0])
colors_eff = plt.cm.RdYlGn(data_dea['Efficiency_Normalized'])

for i, row in data_dea.iterrows():
    ax1.scatter(row['Staff'], row['Revenue'], s=500, c=[colors_eff[i]], 
               marker='o', edgecolors='black', linewidth=2, alpha=0.7)
    ax1.annotate(row['Branch'], (row['Staff'], row['Revenue']), 
                xytext=(5, 5), textcoords='offset points', fontweight='bold', fontsize=10)

ax1.set_xlabel('Input: Staff Count', fontsize=11, fontweight='bold')
ax1.set_ylabel('Output: Revenue ($)', fontsize=11, fontweight='bold')
ax1.set_title('Efficiency Frontier: Staff vs Revenue', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Input vs Cost Efficiency
ax2 = fig.add_subplot(gs[0, 1])
bars = ax2.barh(data_dea['Branch'], data_dea['Revenue_per_Dollar'], 
               color=colors_eff, edgecolor='black', linewidth=2, alpha=0.8)
ax2.set_xlabel('Revenue per Dollar Spent', fontsize=11, fontweight='bold')
ax2.set_title('Cost Efficiency Analysis', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

# Add value labels
for i, (bar, val) in enumerate(zip(bars, data_dea['Revenue_per_Dollar'])):
    ax2.text(val, bar.get_y() + bar.get_height()/2., f'{val:.2f}', 
            ha='left', va='center', fontweight='bold', fontsize=9)

# Efficiency Scores
ax3 = fig.add_subplot(gs[1, 0])
bars_eff = ax3.bar(range(len(data_dea)), data_dea['Efficiency_Normalized'] * 100,
                  color=colors_eff, edgecolor='black', linewidth=2, alpha=0.8)
ax3.axhline(y=np.mean(data_dea['Efficiency_Normalized']) * 100, color='red', 
           linestyle='--', linewidth=2, label=f'Average: {np.mean(data_dea[\"Efficiency_Normalized\"]) * 100:.1f}%')
ax3.set_xticks(range(len(data_dea)))
ax3.set_xticklabels(data_dea['Branch'], rotation=45, ha='right')
ax3.set_ylabel('Efficiency Score (%)', fontsize=11, fontweight='bold')
ax3.set_title('DEA Efficiency Scores', fontsize=12, fontweight='bold')
ax3.set_ylim(0, 110)
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, val in zip(bars_eff, data_dea['Efficiency_Normalized'] * 100):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=9)

# Satisfaction vs Efficiency
ax4 = fig.add_subplot(gs[1, 1])
scatter = ax4.scatter(data_dea['Satisfaction'], data_dea['Efficiency_Normalized'] * 100,
                     s=300, c=data_dea['Revenue'], cmap='viridis', 
                     edgecolors='black', linewidth=2, alpha=0.8)

for i, row in data_dea.iterrows():
    ax4.annotate(row['Branch'], (row['Satisfaction'], row['Efficiency_Normalized'] * 100),
                xytext=(5, 5), textcoords='offset points', fontweight='bold', fontsize=9)

ax4.set_xlabel('Customer Satisfaction Score', fontsize=11, fontweight='bold')
ax4.set_ylabel('Efficiency Score (%)', fontsize=11, fontweight='bold')
ax4.set_title('Efficiency vs Customer Satisfaction', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3)
cbar = plt.colorbar(scatter, ax=ax4)
cbar.set_label('Revenue ($)', fontsize=10, fontweight='bold')

plt.suptitle('Data Envelopment Analysis: Branch Performance Evaluation', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

# Summary statistics
print("✓ Data Envelopment Analysis Complete")
print("\nBranch Efficiency Rankings:")
print(data_dea[['Branch', 'Efficiency_Normalized']].sort_values('Efficiency_Normalized', ascending=False).to_string(index=False))
print(f"\nMean Efficiency: {data_dea['Efficiency_Normalized'].mean():.2%}")
print(f"Std Dev Efficiency: {data_dea['Efficiency_Normalized'].std():.2%}")
print(f"\nMost Efficient: {data_dea.loc[data_dea['Efficiency_Normalized'].idxmax(), 'Branch']}")
print(f"Least Efficient: {data_dea.loc[data_dea['Efficiency_Normalized'].idxmin(), 'Branch']}")

---

## 17. Summary: Models Classification & Comparison

In [ ]:
# Create comprehensive model classification table
models_classification = {
    'DESCRIPTIVE ANALYTICS': [
        {'Model': 'Time Series Decomposition', 'Purpose': 'Separate trend, seasonality, residuals', 'Use Case': 'Pattern identification'},
        {'Model': 'K-Means Clustering', 'Purpose': 'Segment data into groups', 'Use Case': 'Customer segmentation'},
        {'Model': 'Isolation Forests', 'Purpose': 'Detect anomalies/outliers', 'Use Case': 'Fraud/anomaly detection'},
    ],
    'PREDICTIVE ANALYTICS': [
        {'Model': 'Prophet', 'Purpose': 'Time series forecasting', 'Use Case': 'Demand forecasting'},
        {'Model': 'SARIMA', 'Purpose': 'Seasonal time series', 'Use Case': 'Sales prediction'},
        {'Model': 'SARIMAX', 'Purpose': 'Seasonal with external variables', 'Use Case': 'Advanced forecasting'},
        {'Model': 'Holt-Winters', 'Purpose': 'Exponential smoothing', 'Use Case': 'Trend + seasonality'},
        {'Model': 'Holt\\'s Linear Trend', 'Purpose': 'Linear trend forecasting', 'Use Case': 'Growth estimation'},
        {'Model': 'BSTS', 'Purpose': 'Bayesian structural time series', 'Use Case': 'Causal forecasting'},
        {'Model': 'Logistic Regression', 'Purpose': 'Binary classification', 'Use Case': 'Churn prediction'},
        {'Model': 'XGBoost', 'Purpose': 'Ensemble classification/regression', 'Use Case': 'Complex predictions'},
        {'Model': 'Gradient Boosting', 'Purpose': 'Sequential tree boosting', 'Use Case': 'High accuracy prediction'},
        {'Model': 'Markov Chain', 'Purpose': 'State transition modeling', 'Use Case': 'Behavior prediction'},
        {'Model': 'Symbolic Forecasting', 'Purpose': 'Pattern-based forecasting', 'Use Case': 'Symbolic time series'},
    ],
    'PRESCRIPTIVE ANALYTICS': [
        {'Model': 'Linear Programming', 'Purpose': 'Optimize linear objectives', 'Use Case': 'Resource allocation'},
        {'Model': 'MILP', 'Purpose': 'Optimize with integer constraints', 'Use Case': 'Facility location'},
        {'Model': 'Monte Carlo Simulation', 'Purpose': 'Risk analysis via simulation', 'Use Case': 'Project estimation'},
        {'Model': 'Sensitivity Analysis', 'Purpose': 'Parameter impact analysis', 'Use Case': 'Decision support'},
        {'Model': 'Data Envelopment Analysis', 'Purpose': 'Efficiency evaluation', 'Use Case': 'Performance benchmarking'},
    ]
}

# Create visualization
fig = plt.figure(figsize=(18, 12))
gs = GridSpec(3, 1, figure=fig, height_ratios=[1, 1.2, 1])

# Title and categorization explanation
ax_title = fig.add_subplot(gs[0, 0])
ax_title.axis('off')

explanation_text = """
ANALYTICS FRAMEWORK OVERVIEW

DESCRIPTIVE ANALYTICS: Answers "What happened?" by analyzing historical data
   • Time Series Decomposition: Identifies trend, seasonality, and randomness
   • K-Means Clustering: Groups similar entities for better understanding
   • Anomaly Detection: Identifies unusual patterns that deviate from normal

PREDICTIVE ANALYTICS: Answers "What will happen?" by forecasting future outcomes
   • Time Series Models (Prophet, SARIMA, Holt-Winters): Forecast time-dependent data
   • Classification Models (Logistic Regression, XGBoost): Predict categorical outcomes
   • Advanced Models (BSTS, Markov): Incorporate complex temporal relationships
   • Ensemble Methods: Combine multiple models for improved accuracy

PRESCRIPTIVE ANALYTICS: Answers "What should we do?" by recommending optimal actions
   • Optimization (LP, MILP): Find best allocation of resources
   • Simulation (Monte Carlo): Assess risk and uncertainty
   • Sensitivity Analysis: Understand parameter impacts on decisions
   • Efficiency Analysis (DEA): Evaluate and improve performance
"""

ax_title.text(0.05, 0.95, explanation_text, transform=ax_title.transAxes,
             fontsize=10, verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

# Detailed table
ax_table = fig.add_subplot(gs[1, 0])
ax_table.axis('tight')
ax_table.axis('off')

table_data = [['Category', 'Model', 'Purpose/Use Case', 'Key Output']]

for category in models_classification:
    for i, model_info in enumerate(models_classification[category]):
        if i == 0:
            table_data.append([category, model_info['Model'], 
                             f"{model_info['Purpose']}\n({model_info['Use Case']})", 'Insights/Predictions'])
        else:
            table_data.append(['', model_info['Model'], 
                             f"{model_info['Purpose']}\n({model_info['Use Case']})", 'Insights/Predictions'])

table = ax_table.table(cellText=table_data, cellLoc='left', loc='center',
                      colWidths=[0.15, 0.25, 0.4, 0.2])
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2)

# Style header
for i in range(4):
    table[(0, i)].set_facecolor('#2E86AB')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Color code by category
category_colors = {'DESCRIPTIVE ANALYTICS': '#FFE5CC', 'PREDICTIVE ANALYTICS': '#CCE5FF', 'PRESCRIPTIVE ANALYTICS': '#CCFFCC'}
current_category = None

for i in range(1, len(table_data)):
    if table_data[i][0]:  # New category
        current_category = table_data[i][0]
    
    color = category_colors.get(current_category, 'white')
    for j in range(4):
        table[(i, j)].set_facecolor(color)

# Model comparison metrics
ax_metrics = fig.add_subplot(gs[2, 0])
ax_metrics.axis('tight')
ax_metrics.axis('off')

metrics_text = """
KEY PERFORMANCE METRICS USED IN THIS NOTEBOOK

Descriptive Models:
   • Silhouette Score: -1 to 1 (higher = better clustering)
   • Anomaly Detection Rate: % of detected anomalies
   
Predictive Models:
   • MAE (Mean Absolute Error): Average absolute difference from actual
   • RMSE (Root Mean Squared Error): Penalizes larger errors more heavily
   • MAPE (Mean Absolute Percentage Error): % error relative to actual values
   • Accuracy: % of correct predictions
   • F1-Score: Harmonic mean of precision and recall (0-1, higher = better)
   • ROC-AUC: 0 to 1; 0.5 = random, 1.0 = perfect
   
Prescriptive Models:
   • Total Cost/Profit: Objective function value
   • Optimal Solutions: Resource allocation recommendations
   • Risk Metrics (P10, P50, P90): Confidence intervals
   • Efficiency Scores: Performance relative to frontier
"""

ax_metrics.text(0.05, 0.95, metrics_text, transform=ax_metrics.transAxes,
               fontsize=10, verticalalignment='top', fontfamily='monospace',
               bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))

plt.suptitle('Comprehensive Analytics Models Classification Framework', 
            fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print("=" * 80)
print(" " * 20 + "ANALYTICS MODELS CLASSIFICATION SUMMARY")
print("=" * 80)
print("\n✓ ALL MODELS SUCCESSFULLY DEMONSTRATED WITH ACTUAL GRAPHS\n")

print("DESCRIPTIVE ANALYTICS (3 models):")
for model in models_classification['DESCRIPTIVE ANALYTICS']:
    print(f"  • {model['Model']}: {model['Use Case']}")

print("\nPREDICTIVE ANALYTICS (11 models):")
for model in models_classification['PREDICTIVE ANALYTICS']:
    print(f"  • {model['Model']}: {model['Use Case']}")

print("\nPRESCRIPTIVE ANALYTICS (5 models):")
for model in models_classification['PRESCRIPTIVE ANALYTICS']:
    print(f"  • {model['Model']}: {model['Use Case']}")

print("\n" + "=" * 80)
print("NOTEBOOK COMPLETE - Ready for Capstone Project Implementation")
print("=" * 80)